In [1]:
import requests
import json
import os
import pandas as pd
import seaborn as sns
import numpy as np

from random import random
from time import sleep
from scipy import stats
from tqdm.auto import tqdm

In [2]:
S2_API_KEY = os.getenv('S2_API_KEY')

In [3]:
def get_citation_edges(**req_kwargs):
    """This helps with API endpoints that involve paging."""
    page_size = 1000
    offset = 0
    while True:
        req_kwargs.setdefault('params', dict())
        req_kwargs['params']['limit'] = page_size
        req_kwargs['params']['offset'] = offset
        rsp = requests.get(**req_kwargs)
        rsp.raise_for_status()

        page = rsp.json()["data"]
        if not page:
            break
        for element in page:
            yield element

        if len(page) < page_size:
            break  # no more pages
        offset += page_size

def get_paper(paper_id):
    rsp = requests.get(f'https://api.semanticscholar.org/graph/v1/paper/{paper_id}',
                       headers={'X-API-KEY': S2_API_KEY},
                       params={'fields': 'title,publicationVenue,externalIds,publicationDate,isOpenAccess,openAccessPdf'})
    rsp.raise_for_status()
    return rsp.json()

def get_paper_bulk(paper_ids):
    rsp = requests.post(
        'https://api.semanticscholar.org/graph/v1/paper/batch',
        headers={'X-API-KEY': S2_API_KEY},
        params={'fields': 'title,publicationVenue,externalIds,publicationDate,isOpenAccess,openAccessPdf'},
        json={'ids': paper_ids}
    )
    rsp.raise_for_status()
    return rsp.json()        

def get_references(paper_id):
    edges = get_citation_edges(url=f'https://api.semanticscholar.org/graph/v1/paper/{paper_id}/references',
                               headers={'X-API-KEY': S2_API_KEY},
                               params={'fields': 'title,publicationDate,isOpenAccess,openAccessPdf'})
    return list(edge['citedPaper'] for edge in edges)

In [4]:
dpo_paper = '0d1c76d45afa012ded7ab741194baf142117c495'

In [13]:
def paper_bfs(curr, hops_left=0): 
    if hops_left:
        try: 
            curr_refs = get_references(curr)
        except requests.HTTPError: 
            # Wait on 429 too many requests
            sleep(1+2*random())
            curr_refs = get_references(curr)
        refs_with_pdf = [d for d in curr_refs if d['isOpenAccess']]
        curr_subtree = []
        for d in refs_with_pdf:
            curr_subtree += paper_bfs(d['paperId'], hops_left-1)
        return [curr] + curr_subtree
    else:
        return [curr]

In [14]:
dpo_tree = paper_bfs(dpo_paper, 2)

In [16]:
dpo_tree_papers = get_paper_bulk(dpo_tree)
dpo_tree_papers = [d for d in dpo_tree_papers if d['publicationDate']]

In [18]:
papers_df = pd.DataFrame(dpo_tree_papers).set_index('paperId')
# papers_df['url'] = papers_df.map(lambda r: json.loads(r.openAccessPdf

In [20]:
pub_dates = pd.to_datetime(pd.Series(
    data=[d['publicationDate'] for d in dpo_tree_papers],
    index=[d['paperId'] for d in dpo_tree_papers],
    name='pub_date',
))
pub_dates

0d1c76d45afa012ded7ab741194baf142117c495   2023-05-29
be55e8ec4213868db08f2c3168ae666001bea4b8   2023-04-03
deb8f26509ae320fc975b32922416cb156c61bbd   2023-04-21
762ca2711eb167f19b79e39c175708ca15e1f5d7   2023-03-14
16c64f74ce0e6a59b0709c0d8e66596a5bc08ed6   2023-03-07
                                              ...    
10468b38072f70cad77109441c73f5f470da33f9   2012-09-01
bc6dff14a130c57a91d5a21339c23471faf1d46f   2008-10-24
1f869232f148ec52066fab06a49855937f84098b   2007-06-20
f8222304ca201f8d524b3aa270673023334e7ed1   1979-09-25
d244943df2e046157057220e46b64a868fed3549   1960-07-01
Name: pub_date, Length: 298, dtype: datetime64[ns]

In [48]:
pub_dates.to_csv('dpo_tree_pub_dates.csv')

In [12]:
# recipricoal frequency sampling
min_date = pub_dates.min()
float_dates = (pub_dates - min_date) / np.timedelta64(1, 'D')
dist = stats.gaussian_kde(float_dates)
recipricoal_weights = 1 / (dist(float_dates) + 1e-10)

In [28]:
# samp = pub_dates.sample(n=100, weights=recipricoal_weights)
samp = pub_dates.reset_index()

In [23]:
import urllib.request

In [35]:
dpo_tree_papers[0]

{'paperId': '0d1c76d45afa012ded7ab741194baf142117c495',
 'externalIds': {'DBLP': 'conf/nips/RafailovSMMEF23',
  'ArXiv': '2305.18290',
  'DOI': '10.48550/arXiv.2305.18290',
  'CorpusId': 258959321},
 'publicationVenue': {'id': 'd9720b90-d60b-48bc-9df8-87a30b9a60dd',
  'name': 'Neural Information Processing Systems',
  'type': 'conference',
  'alternate_names': ['Neural Inf Process Syst', 'NeurIPS', 'NIPS'],
  'url': 'http://neurips.cc/'},
 'title': 'Direct Preference Optimization: Your Language Model is Secretly a Reward Model',
 'isOpenAccess': True,
 'openAccessPdf': {'url': 'http://arxiv.org/pdf/2305.18290',
  'status': 'CLOSED',
  'license': None},
 'publicationDate': '2023-05-29'}

In [36]:
import arxiv 

In [37]:
client = arxiv.Client()

In [38]:
paper = next(arxiv.Client().results(arxiv.Search(id_list=["1605.08386v1"])))

In [53]:
[paper['externalIds']['ArXiv'] for paper in dpo_tree_papers if 'ArXiv' in paper['externalIds']]

['2305.18290',
 '2304.01373',
 '2304.11158',
 '2303.08112',
 '2303.03915',
 '2212.10511',
 '2212.09803',
 '2211.13709',
 '2211.08411',
 '2210.15424',
 '2210.06413',
 '2210.02414',
 '2209.03661',
 '2207.14251',
 '2207.10245',
 '2207.00099',
 '2205.12628',
 '2205.10770',
 '2205.10487',
 '2205.06266',
 '2206.03216',
 '2204.13509',
 '2204.08583',
 '2204.06745',
 '2204.06125',
 '2204.05999',
 '2203.15395',
 '2202.13169',
 '2112.10752',
 '2111.09259',
 '2109.10052',
 '2109.03858',
 '2107.06499',
 '2106.15590',
 '2104.09864',
 '2104.08758',
 '2103.12028',
 '2103.07853',
 '2010.14571',
 '2010.00133',
 '1911.02116',
 '1904.03035',
 '1804.06876',
 '1707.09457',
 '1606.06031',
 '2302.08215',
 '2210.11416',
 '2209.14375',
 '2207.10342',
 '2206.11684',
 '2206.05802',
 '2206.00761',
 '2205.11275',
 '2205.01663',
 '2204.05862',
 '2203.11147',
 '2109.07445',
 '2109.07958',
 '2106.01465',
 '2102.09130',
 '2009.11462',
 '2005.00661',
 '2004.13637',
 '1909.01326',
 '1811.10996',
 '1809.10736',
 '1805.048

In [54]:
paper = dpo_tree_papers[0]
print(paper['openAccessPdf']['url'])
urllib.request.urlretrieve(paper['openAccessPdf']['url'], f'corpus/{paper["paperId"]}.pdf')

http://arxiv.org/pdf/2305.18290


HTTPError: HTTP Error 403: Forbidden

In [56]:
urllib.request.urlretrieve('http://export.arxiv.org/pdf/2210.10863.pdf')

HTTPError: HTTP Error 403: Forbidden

In [49]:
no_pdf = []
for idx in tqdm(samp.index):
    paper = dpo_tree_papers[idx]
    sleep(1.1)
    try: 
        if paper['externalIds']['ArXiv']:
            arxiv_paper = next(arxiv.Client().results(arxiv.Search(id_list=["1605.08386v1"])))
            arxiv_paper.download_pdf(filename=f'corpus/{paper["paperId"]}.pdf')
        else: 
            urllib.request.urlretrieve(paper['openAccessPdf']['url'], f'corpus/{paper["paperId"]}.pdf')
    except urllib.request.HTTPError as e : 
        no_pdf.append(paper['paperId'])
        print(e.code, e.url)
        raise e

  0%|          | 0/298 [00:00<?, ?it/s]

403 http://export.arxiv.org/pdf/1605.08386v1


HTTPError: HTTP Error 403: Forbidden

In [43]:
samp

297   1960-07-01
290   2004-07-01
205   2000-03-01
296   1979-09-25
284   2015-05-03
         ...    
219   2017-08-01
226   2015-09-22
285   2014-12-05
67    2018-09-27
9     2022-10-27
Length: 100, dtype: datetime64[ns]